# Nova's first conversation: prompted Qwen3.5-2B through the agent loop

Serves the base Qwen3.5-2B with vLLM (the setup that passed the GPU risk tests), then runs one customer conversation
through `agent/loop.py` with prompt v0 and the six rough tools in `agent/tools_v0.py`. The tools query the
store's database on Neon. Nothing is trained here: this is the floor that fine-tuning has to beat.

**Before running (Kaggle):** Settings → Accelerator **GPU T4 x2** (one T4 is used), Internet **On**;
Add-ons → Secrets → add **`NEON_URL`** (the Neon connection string, pooling off) and tick it; tick `HF_TOKEN` if you have it.
Then **Run All** (first vLLM start takes about 10 minutes). Copy everything from `===== CONVERSATION REPORT =====` to the end.


In [ ]:
# 1. Secrets. The database URL comes from Kaggle's secrets panel, never from this notebook.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["DB_URL"] = secrets.get_secret("NEON_URL")
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")   # optional: avoids download rate limits
except Exception:
    pass
print("DB host:", os.environ["DB_URL"].split("@")[-1].split("/")[0])


In [ ]:
# 2. Clone the repo; the agent needs only openai and psycopg in this kernel.
import subprocess, sys
REPO_DIR = os.path.abspath("customer-support-SLM")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jagjotsingh244-source/customer-support-SLM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
%pip install -q openai "psycopg[binary]"
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

from agent.tools_v0 import Session, connect     # fail fast if the database is unreachable
print("database ok, customer 416 is", Session.open(connect(os.environ["DB_URL"]), 416).customer_name)


In [ ]:
# 3. vLLM in its own environment, pinned to the versions that passed the risk tests.
!uv --version || pip install -q uv
!uv venv -q /tmp/vllm-env --python 3.12
!uv pip install -q --python /tmp/vllm-env/bin/python -r requirements-serve.txt


In [ ]:
# 4. Start vLLM (T4 libcuda fix first) and wait until it answers.
import time, urllib.request
FLAGS = ("--dtype half --max-model-len 8192 --language-model-only "
         "--enable-auto-tool-choice --tool-call-parser qwen3_coder --port 8000")
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(["bash", "-c", f"source serving/t4_libcuda_fix.sh && "
                           f"exec /tmp/vllm-env/bin/vllm serve Qwen/Qwen3.5-2B {FLAGS}"],
                          stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1500:
            print(open("/kaggle/working/vllm.log").read()[-4000:])
            raise RuntimeError("vLLM did not start; full log in /kaggle/working/vllm.log")
        time.sleep(10)
print(f"vLLM up after {time.time() - t0:.0f}s")


In [ ]:
# 5. One conversation, every message printed: the delayed-order case for eval customer 416.
#    Order 11904 (webcam) had an estimated delivery date of Monday 16 November and is stuck at Nagpur hub.
MESSAGE = "hi where is my order?? it was supposed to come by monday"
!python -m agent.chat --base-url http://localhost:8000/v1 --model Qwen/Qwen3.5-2B --customer 416 \
    --out /kaggle/working/chat-2b.json "{MESSAGE}"


In [ ]:
# 6. Latency: the same conversation 5 more times (warm server), median and p95 of the whole turn.
import statistics, time
from openai import OpenAI
from agent.loop import run_turn
from agent.prompts import system_prompt
from agent.tools_v0 import REGISTRY, TOOLS
from config import now

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
session = Session.open(connect(os.environ["DB_URL"]), 416)
RUNS = []
for i in range(5):
    msgs = [{"role": "system", "content": system_prompt(now(), session.customer_name)},
            {"role": "user", "content": MESSAGE}]
    t0 = time.perf_counter()
    reply, trace = run_turn(client, "Qwen/Qwen3.5-2B", msgs, TOOLS, REGISTRY, session)
    RUNS.append({"turn_s": time.perf_counter() - t0,
                 "model_s": sum(c["latency_s"] for c in trace["model_calls"]),
                 "tools": [(c["name"], c["args"]) for c in trace["tool_calls"]],
                 "reply": reply})
    print(f"run {i + 1}: {RUNS[-1]['turn_s']:.2f}s  tools={RUNS[-1]['tools']}\n   {reply}")


In [ ]:
# 7. REPORT: copy everything from the next line to the end and paste it back.
import json
print("===== CONVERSATION REPORT =====")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
chat = json.load(open("/kaggle/working/chat-2b.json"))
print("\n--- first conversation ---")
for m in chat["messages"]:
    if m["role"] == "tool":
        print("[tool result]", m["content"][:800])
    else:
        for c in m.get("tool_calls", []):
            print(f"[{m['role']} -> tool] {c['function']['name']}({c['function']['arguments']})")
        if m.get("content"):
            print(f"[{m['role']}]", m["content"])
print("\nmodel calls:", chat["traces"][0]["model_calls"])
turns = sorted(r["turn_s"] for r in RUNS)
print("\n--- 5 more runs ---")
for r in RUNS:
    print(f"{r['turn_s']:.2f}s (model {r['model_s']:.2f}s) tools={r['tools']}\n   {r['reply']}")
print(f"\nturn latency: median {statistics.median(turns):.2f}s, p95 (max of 5) {turns[-1]:.2f}s")
print("===== END OF REPORT =====")
